# Study 917 — Stale NAV — the teardown

Lagged HAC regressions per fund (raw, net of SPY's own next-day move at a unit hedge, and at a fitted contemporaneous beta), the confound decomposition, the domestic SPY-on-SPY control, the top-decile next-day rule on an equal-weight basket, the block bootstrap, the 2010 era cut, the cost / borrow / threshold sweeps, the conservative extra-lag variant, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `42e35a0143ca`, as-of 2026-06-30).

In [1]:
R = {'start': '1993-01-29', 'end': '2026-06-30', 'n_days': 8411, 'fp': '42e35a0143ca', 'fund_start': '1996-03-18', 'fxi_start': '2004-10-08', 'n_long': 7619, 'n_fxi': 5464, 'ewj': (-0.111, -5.85, -0.027, -1.51), 'ewg': (-0.06, -2.25, 0.024, 1.58), 'fxi': (-0.263, -6.54, -0.16, -4.3), 'ewa': (-0.073, -1.75, 0.011, 0.45), 'ewu': (-0.056, -2.13, 0.028, 2.01), 'dom_beta': -0.082, 'dom_t': -3.51, 'dom_n': 8409, 'bonferroni': 2.58, 'hed_ewj': (0.754, 0.56, -0.048, -3.19), 'hed_ewg': (0.993, 1.37, 0.023, 1.54), 'hed_fxi': (1.161, 0.36, -0.143, -3.65), 'hed_ewa': (0.921, 1.04, 0.004, 0.17), 'hed_ewu': (0.866, 1.28, 0.017, 1.2), 'rel_bps_hedged': -1.55, 'rel_t_hedged': -0.46, 'basket_beta': 0.906, 'era_ewj': (-0.059, -2.98, 0.015, 0.56), 'era_fxi': (-0.347, -7.31, -0.046, -1.54), 'era_ewa': (0.063, 2.02, -0.057, -1.68), 'erah_ewj': (-0.076, -3.85, -0.01, -0.55), 'erah_fxi': (-0.288, -5.63, -0.055, -1.9), 'erah_ewu': (0.029, 1.4, 0.002, 0.12), 'n_on': 819, 'trig_frac': 10.7, 'turnover': 1402, 'on_bps': -4.92, 'on_t': -0.82, 'off_bps': 3.45, 'diff_bps': -8.37, 'diff_t': -1.41, 'rel_bps': -1.3, 'rel_t': -0.39, 'long_sh': -0.701, 'long_cagr': -6.15, 'long_vol': 8.52, 'long_t': -3.77, 'short_sh': -0.419, 'short_t': -2.32, 'bh_sh': 0.298, 'bh_cagr': 4.19, 'bh_vol': 21.51, 'bh_t': 1.89, 'boot_lo': -19.75, 'boot_hi': 6.63, 'boot_neg': 77.3, 'era_e_n': 3472, 'era_e_on': 3.95, 'era_e_t': 0.53, 'era_e_long': -0.447, 'era_e_bh': 0.279, 'era_l_n': 4146, 'era_l_on': -13.17, 'era_l_t': -1.58, 'era_l_long': -0.927, 'era_l_bh': 0.323, 'cost0_long': -0.157, 'cost0_t': -0.86, 'cost0_short': 0.127, 'cost0_short_t': 0.7, 'cost5_short': -0.146, 'cost25_long': -1.494, 'borrow0': -0.419, 'borrow2': -0.444, 'borrow5': -0.482, 'thr20': (-3.13, -0.87), 'thr10': (-4.92, -0.82), 'thr5': (-7.02, -0.72), 'thr1': (-60.57, -2.18), 'thr1_n': 109, 'lag2_bps': 0.7, 'lag2_t': 0.13, 'bil_on': -28.56, 'bil_t': -2.59, 'bil_long': -1.037, 'bil_bh': 0.244, 'syn_planted': 0.25, 'syn_rec': 0.249, 'syn_t': 18.0, 'syn_on': 53.6, 'syn_on_t': 11.5, 'syn_null_beta': -0.004, 'syn_null_sd': 0.007, 'syn_null_fire': 1}

## 1. Specification

`r_fund[t+1] = a + b · r_SPY[t]`, Newey-West SEs with the Andrews-style automatic truncation `floor(4·(n/100)^(2/9))`. Total-return closes (`auto_adjust=True`) — country ETFs distribute lumpily and a price-only tape injects a sawtooth into the regressand. Cash is `^IRX` compounded daily and lagged one day (a **PROXY**; BIL cross-checks 2007+). **Exactly one execution lag**: the signal is complete at the close of `t` and the position is on for the `t+1` return.

> 💡 **In plain words** — did today's Wall Street session leave tomorrow's Tokyo fund with money owing?

In [2]:
print('fund   n      raw b     t      net-of-SPY b     t')
for k, nm in [('ewj','EWJ'), ('ewg','EWG'), ('fxi','FXI'), ('ewa','EWA'), ('ewu','EWU')]:
    b, t, br, tr = R[k]
    n = R['n_fxi'] if k == 'fxi' else R['n_long']
    print(f"{nm:5s} {n:6d} {b:+8.3f} {t:+7.2f} {br:+13.3f} {tr:+8.2f}")
print()
print(f"domestic control  SPY[t+1] ~ SPY[t]: b={R['dom_beta']:+.3f} "
      f"(t={R['dom_t']:+.2f}, n={R['dom_n']})")
print(f"Bonferroni bar across 5 funds: |t| >= {R['bonferroni']:.2f}")

fund   n      raw b     t      net-of-SPY b     t
EWJ     7619   -0.111   -5.85        -0.027    -1.51
EWG     7619   -0.060   -2.25        +0.024    +1.58
FXI     5464   -0.263   -6.54        -0.160    -4.30
EWA     7619   -0.073   -1.75        +0.011    +0.45
EWU     7619   -0.056   -2.13        +0.028    +2.01

domestic control  SPY[t+1] ~ SPY[t]: b=-0.082 (t=-3.51, n=8409)
Bonferroni bar across 5 funds: |t| >= 2.58


## 2. The confound, and the ASSUMPTION hiding inside the control

SPY reverses on itself at *b* = -0.082 (*t* = -3.51), and a country ETF loads that through its US beta. The relative regression `(r_fund[t+1] − r_SPY[t+1]) = a + b·r_SPY[t]` nets it out — but subtracting **one** unit of SPY assumes β = 1 exactly. It is not: β runs 0.75 (EWJ) to 1.16 (FXI), so the unit hedge leaves `(β−1)` units of SPY in the residual and re-imports `(β−1)·-0.082` of the domestic reversal.

The cell below drops the assumption: β is refit on the same sample and the fund is hedged at that ratio, giving the exact identity `b_raw = β·b_domestic + b_hedged`. **CAVEAT: that β is fitted in-sample and applied in-sample** — a diagnostic, never a tradable rule, and it stamps no badge. It is here because it is load-bearing, and hiding it would be dishonest.

Two corrections follow. (i) The confound does **not** uniformly explain "most" of the raw slopes: it *over*-explains EWG/EWA/EWU (104%–137%, i.e. those funds reverse *less* than their beta implies) but covers only 56% of EWJ and 36% of FXI. (ii) Under the fitted hedge **two** funds clear |*t*| ≥ 2.58, not one — EWJ (-3.19) joins FXI (-3.65) — and **both are negative**. No fund is positive under either hedge. Correcting the hedge ratio sharpens the *reversal*; it never resurrects the catch-up.

In [3]:
print('beta_c   raw_b  =  confound (share)  +  hedged_b (t)   [fitted beta, IN-SAMPLE]')
for k, nm in [('ewj','EWJ'), ('ewg','EWG'), ('fxi','FXI'), ('ewa','EWA'), ('ewu','EWU')]:
    bc, share, bh, th = R['hed_' + k]
    raw = R[k][0]
    print(f"{nm:4s} {bc:6.3f} {raw:+8.3f}  =  {bc*R['dom_beta']:+.3f} ({share:5.0%})"
          f"  +  {bh:+.3f} (t={th:+.2f})")
print()
print('net-of-SPY slope, split at 2010-01-01 — unit hedge [fitted hedge, beta refit per era]')
for k, h, nm in [('era_ewj','erah_ewj','EWJ'), ('era_fxi','erah_fxi','FXI'),
                 ('era_ewa',None,'EWA')]:
    be, te, bl, tl = R[k]
    tail = ''
    if h:
        hbe, hte, hbl, htl = R[h]
        tail = f"   [{hbe:+.3f} (t={hte:+.2f}) -> {hbl:+.3f} (t={htl:+.2f})]"
    print(f"  {nm}: early {be:+.3f} (t={te:+.2f})   late {bl:+.3f} (t={tl:+.2f}){tail}")
print('\nBoth Bonferroni-clearing slopes are pre-2010 and both are NEGATIVE:')
print('after 2010 no fund clears |t|=2 in either hedge, in either direction.')

beta_c   raw_b  =  confound (share)  +  hedged_b (t)   [fitted beta, IN-SAMPLE]
EWJ   0.754   -0.111  =  -0.062 (  56%)  +  -0.048 (t=-3.19)
EWG   0.993   -0.060  =  -0.081 ( 137%)  +  +0.023 (t=+1.54)
FXI   1.161   -0.263  =  -0.095 (  36%)  +  -0.143 (t=-3.65)
EWA   0.921   -0.073  =  -0.076 ( 104%)  +  +0.004 (t=+0.17)
EWU   0.866   -0.056  =  -0.071 ( 128%)  +  +0.017 (t=+1.20)

net-of-SPY slope, split at 2010-01-01 — unit hedge [fitted hedge, beta refit per era]
  EWJ: early -0.059 (t=-2.98)   late +0.015 (t=+0.56)   [-0.076 (t=-3.85) -> -0.010 (t=-0.55)]
  FXI: early -0.347 (t=-7.31)   late -0.046 (t=-1.54)   [-0.288 (t=-5.63) -> -0.055 (t=-1.90)]
  EWA: early +0.063 (t=+2.02)   late -0.057 (t=-1.68)

Both Bonferroni-clearing slopes are pre-2010 and both are NEGATIVE:
after 2010 no fund clears |t|=2 in either hedge, in either direction.


## 3. The tradable rule — equal-weight basket, excess-of-cash

Trigger = SPY's day-`t` return in the top decile of its own **expanding** history (min 250 obs, so the cut is never chosen with hindsight). Cost 10 bps one-way × NAV per position change; no short leg on the long arm, hence no borrow.

Three disclosures. The trigger-day mean below is **gross of trading cost** (the claim's best case — cost enters only the Sharpe rows). Costs are **one-sided**: the rule pays on all 1,402 position changes, buy-and-hold pays nothing, an asymmetry worth ~2 bps in total over 30 years that runs *against* the claim. And the HAC *t* on trigger-day means sits on a **non-contiguous subsample**, so the era cut and the block bootstrap — which respect calendar time — are what the verdict leans on.

> 💡 **In plain words** — buy the foreign funds the day after Wall Street rips.

In [4]:
print(f"trigger days {R['n_on']} ({R['trig_frac']:.1f}%), {R['turnover']} position changes")
print(f"trigger-day basket excess return {R['on_bps']:+.2f} bps (HAC t={R['on_t']:+.2f})"
      f"   [GROSS of cost]")
print(f"all other days                   {R['off_bps']:+.2f} bps  "
      f"| difference {R['diff_bps']:+.2f} bps (Welch t={R['diff_t']:+.2f})")
print(f"NET OF SPY same-day move         {R['rel_bps']:+.2f} bps (HAC t={R['rel_t']:+.2f})"
      f"   [unit-beta ASSUMPTION]")
print(f"  same at the fitted beta {R['basket_beta']:.3f}   {R['rel_bps_hedged']:+.2f} bps "
      f"(HAC t={R['rel_t_hedged']:+.2f})   [assumption dropped — same answer]")
print()
print(f"long rule   : exSharpe {R['long_sh']:+.3f}  CAGR {R['long_cagr']:+.2f}%  "
      f"vol {R['long_vol']:.1f}%  HAC t {R['long_t']:+.2f}")
print(f"short mirror: exSharpe {R['short_sh']:+.3f}  (t {R['short_t']:+.2f})")
print(f"buy & hold  : exSharpe {R['bh_sh']:+.3f}  CAGR {R['bh_cagr']:+.2f}%  "
      f"vol {R['bh_vol']:.1f}%  HAC t {R['bh_t']:+.2f}")
print(f"\nblock bootstrap (2000 draws, 21-day blocks) on the trigger-day mean: "
      f"95% CI [{R['boot_lo']:+.2f}, {R['boot_hi']:+.2f}] bps, {R['boot_neg']:.1f}% of draws < 0")

trigger days 819 (10.7%), 1402 position changes
trigger-day basket excess return -4.92 bps (HAC t=-0.82)   [GROSS of cost]
all other days                   +3.45 bps  | difference -8.37 bps (Welch t=-1.41)
NET OF SPY same-day move         -1.30 bps (HAC t=-0.39)   [unit-beta ASSUMPTION]
  same at the fitted beta 0.906   -1.55 bps (HAC t=-0.46)   [assumption dropped — same answer]

long rule   : exSharpe -0.701  CAGR -6.15%  vol 8.5%  HAC t -3.77
short mirror: exSharpe -0.419  (t -2.32)
buy & hold  : exSharpe +0.298  CAGR +4.19%  vol 21.5%  HAC t +1.89

block bootstrap (2000 draws, 21-day blocks) on the trigger-day mean: 95% CI [-19.75, +6.63] bps, 77.3% of draws < 0


## 4. Era cut, cost sweep, borrow sweep, threshold sweep, extra lag

In [5]:
print(f"1996-2009 (n={R['era_e_n']}): trigger {R['era_e_on']:+.2f} bps "
      f"(t={R['era_e_t']:+.2f})  long {R['era_e_long']:+.3f}  B&H {R['era_e_bh']:+.3f}")
print(f"2010-2026 (n={R['era_l_n']}): trigger {R['era_l_on']:+.2f} bps "
      f"(t={R['era_l_t']:+.2f})  long {R['era_l_long']:+.3f}  B&H {R['era_l_bh']:+.3f}")
print()
print(f"cost 0 bps : long {R['cost0_long']:+.3f} (t={R['cost0_t']:+.2f})  "
      f"short {R['cost0_short']:+.3f} (t={R['cost0_short_t']:+.2f})  <- only positive cell")
print(f"cost 5 bps : short {R['cost5_short']:+.3f}   cost 25 bps: long {R['cost25_long']:+.3f}")
print(f"borrow on the short mirror: 0% {R['borrow0']:+.3f}  2% {R['borrow2']:+.3f}  "
      f"5% {R['borrow5']:+.3f}  (the spread killed it first)")
print()
for lbl, k in [('top 20%','thr20'), ('top 10%','thr10'), ('top 5%','thr5'), ('top 1%','thr1')]:
    m, t = R[k]
    print(f"  {lbl}: {m:+7.2f} bps (t={t:+.2f})")
print(f"  (the top-1% cell is {R['thr1_n']} crisis days and still the WRONG sign)")
print(f"\nwait one extra day before trading: {R['lag2_bps']:+.2f} bps (t={R['lag2_t']:+.2f})")
print(f"BIL cash cross-check 2007+: trigger {R['bil_on']:+.2f} bps (t={R['bil_t']:+.2f}), "
      f"long {R['bil_long']:+.3f} vs B&H {R['bil_bh']:+.3f}")

1996-2009 (n=3472): trigger +3.95 bps (t=+0.53)  long -0.447  B&H +0.279
2010-2026 (n=4146): trigger -13.17 bps (t=-1.58)  long -0.927  B&H +0.323

cost 0 bps : long -0.157 (t=-0.86)  short +0.127 (t=+0.70)  <- only positive cell
cost 5 bps : short -0.146   cost 25 bps: long -1.494
borrow on the short mirror: 0% -0.419  2% -0.444  5% -0.482  (the spread killed it first)

  top 20%:   -3.13 bps (t=-0.87)
  top 10%:   -4.92 bps (t=-0.82)
  top 5%:   -7.02 bps (t=-0.72)
  top 1%:  -60.57 bps (t=-2.18)
  (the top-1% cell is 109 crisis days and still the WRONG sign)

wait one extra day before trading: +0.70 bps (t=+0.13)
BIL cash cross-check 2007+: trigger -28.56 bps (t=-2.59), long -1.037 vs B&H +0.244


## 5. Live synthetic control — power on, false positives off

Planted world: `r_f[t] = drift + 0.55·r_us[t] + 0.25·r_us[t-1] + eps`. Null world: the same contemporaneous 0.55 load, zero lagged term — identical correlation structure *today*, nothing owed *tomorrow*. The detector must separate them.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from stale_nav import data, strategy as st
p1, t1 = data.synthetic_panel(signal_strength=1.0, seed=917)
d1 = st.synthetic_detect(p1, t1)
print('planted %+.3f -> recovered %+.3f (mean HAC t %+.1f); trigger-day mean %+.1f bps (t %+.1f)'
      % (d1['planted_beta'], d1['beta_mean'], d1['t_beta_mean'],
         d1['mean_on_bps'], d1['t_on_mean']))
nulls = [st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0, seed=917+s))
         for s in range(8)]
nb = np.array([d['beta_mean'] for d in nulls])
nt = np.array([d['t_beta_max_abs'] for d in nulls])
print('null x8: beta %+.4f (sd %.4f); max |t| across 5 funds >= 2 on %d/8 seeds'
      % (nb.mean(), nb.std(ddof=1), int((nt >= 2).sum())))

planted +0.250 -> recovered +0.249 (mean HAC t +18.0); trigger-day mean +53.6 bps (t +11.5)


null x8: beta -0.0041 (sd 0.0069); max |t| across 5 funds >= 2 on 1/8 seeds


## Verdict

- **Signal — None.** No catch-up anywhere, under either hedge: all five raw slopes are negative, and the domestic control (-0.082, *t* = -3.51) accounts for part of that — all of EWG/EWA/EWU, but only 56% of EWJ and 36% of FXI. Net of it, **no fund has a positive timezone slope**; the largest positive in the study is EWG at *t* = +1.54, against a Bonferroni bar of 2.58. What clears that bar clears it **backwards** — FXI (-4.30 unit, -3.65 fitted) and, once the unit-beta shortcut is dropped, EWJ (-3.19) — and both die at the era cut (-1.90 and -0.55 after 2010), so even the anti-claim is a pre-2010 artifact. On the basket the timezone-specific trigger-day return is -1.30 bps (*t* = -0.39) — -1.55 (-0.46) at the fitted β — CI [-19.75, +6.63], sign-flipping across eras, and +0.70 bps (*t* = +0.13) with one extra day of delay. A **Real** stamp needs a robust |*t*| ≥ 2 in the *claimed* direction on this tape; the claimed direction never reaches |*t*| = 1.6 in any specification run here. The synthetic control recovers a planted +0.250 as +0.249 (*t* = +18.0) and stays at -0.0040 on the null, so this is a genuine absence, not a blind harness.
- **Tradability — Mirage.** Long: excess Sharpe -0.701 net versus +0.298 for holding the same basket — you pay two spreads on 10.7% of days for a slightly-worse-than-average session. Short mirror: +0.127 **gross** (*t* = +0.70), negative by 5 bps one-way, before borrow, on five ETFs that are not cheap to short.
- **Scope.** Close-to-close on US-listed ETFs, which price the US session as it happens. The 1990s stale-NAV trade fed on a once-a-day mutual-fund strike that no longer exists; this is the modern-wrapper answer, not a refutation of that literature.